# DART + V6 compacto
¿Una rama de señal de bajo coste conserva la mejora del V6 del compañero? El referente en la primera ronda fue **83,64 % de F1 macro** con la rama completa y **81,94 %** con DART. El experimento compacto conserva las mismas cinco particiones por persona, la mezcla 75/25, el contexto y el HMM. La comparación principal fijada antes de entrenar es **DART + compacto con HMM**. También se mide la contribución del HMM.

La señal de cada noche procede del preprocesamiento V6 de la comparación justa (100 Hz, 0,3–35 Hz, normalización por época y canal). Se resume cada época de 30 s con seis ventanas de 5 s y seis bandas de potencia por canal. La CRNN aprende una representación de 32 dimensiones por época en bloques de 15 épocas. Un LightGBM usa la representación anterior, actual y siguiente. El HMM calcula la probabilidad de cada etapa usando toda la noche. Este protocolo corresponde a inferencia retrospectiva. El cabezal se ajusta sobre representaciones de las mismas personas usadas para ajustar el extractor, como en V6. Las personas de prueba quedan fuera de ambos ajustes.

El objetivo de coste es **20 minutos por ajuste**. Se cronometra la rama completa desde las entradas compactas preparadas y se informa por separado de su preparación, del ajuste DART histórico y de la inferencia. Se reutilizan los DART ya entrenados en cada ronda. Para reproducir, generar las cachés de señal y los modelos DART descritos en [agent.md](../agent.md).

**Resultado de las cinco rondas.** DART + V6 compacto con HMM obtuvo **80,96 % de F1 macro** frente a **80,10 %** de DART y **80,86 %** de DART + CRNN pequeña. En Telemetry el compacto obtuvo 79,90 % frente al 79,30 % de la combinación pequeña. En Cassette obtuvo 77,89 % frente al 79,49 %. La combinación pequeña conservó mayor exactitud y kappa globales. En la primera ronda, el V6 completo combinado obtuvo **83,64 %**, el compacto combinado **82,84 %**. El ajuste en serie de DART y la rama compacta sumó **7 min 16 s** desde entradas preparadas en esa ronda.

In [1]:
import os
import sys
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'data.dvc').is_file())
sys.path.insert(0, str(root))
os.environ['TF_NUM_INTRAOP_THREADS'] = '5'
os.environ['TF_NUM_INTEROP_THREADS'] = '2'
from scripts.evaluar_v6_compacto import preparar, evaluar, estadisticas, JUSTA, CLASES
import json
import pandas as pd
from sklearn.metrics import confusion_matrix
control, dart, noches, s_dart, s_senal = preparar()
display(control.groupby('cohorte').agg(personas=('subject_id','nunique'), noches=('record_id','nunique'), segmentos=('real','size')))
print(f'Lectura tabular e inferencia DART para las 5 rondas: {s_dart:.1f} s. Preparación espectral de las 56 noches desde las señales V6 guardadas: {s_senal:.1f} s.')
resultados, tiempos = [], []

,personas,noches,segmentos
cohorte,,,
Cassette,12,12,32921
Telemetry,22,44,42691


Lectura tabular e inferencia DART para las 5 rondas: 8.9 s. Preparación espectral de las 56 noches desde las señales V6 guardadas: 7.1 s.


## Comparación por ronda

Cada fila de prueba corresponde a una persona ausente del entrenamiento. Las tablas muestran el total conjunto de segmentos evaluados en cada ronda.

In [2]:
m, t = evaluar(1, control, dart, noches)
resultados.append(m)
tiempos.append(t)
display(estadisticas(m).round(4))
print(f"Ronda 1 · {m.subject_id.nunique()} personas · {len(m)} segmentos de prueba · ajuste de la rama {t['ajuste_rama_s']:.1f} s")

,F1 macro,Exactitud,Kappa
Modelo,,,
DART,0.8194,0.8745,0.8262
DART + CRNN,0.8274,0.8805,0.8343
V6 compacto,0.7439,0.8164,0.7477
DART + compacto sin HMM,0.8260,0.8806,0.8347
DART + compacto con HMM,0.8284,0.8844,0.8395


Ronda 1 · 7 personas · 15893 segmentos de prueba · ajuste de la rama 15.0 s


In [3]:
m, t = evaluar(2, control, dart, noches)
resultados.append(m)
tiempos.append(t)
display(estadisticas(m).round(4))
print(f"Ronda 2 · {m.subject_id.nunique()} personas · {len(m)} segmentos de prueba · ajuste de la rama {t['ajuste_rama_s']:.1f} s")

,F1 macro,Exactitud,Kappa
Modelo,,,
DART,0.8025,0.8765,0.8249
DART + CRNN,0.8155,0.8876,0.8397
V6 compacto,0.7836,0.8660,0.8096
DART + compacto sin HMM,0.8122,0.8833,0.8342
DART + compacto con HMM,0.8167,0.8878,0.8403


Ronda 2 · 7 personas · 15763 segmentos de prueba · ajuste de la rama 14.8 s


In [4]:
m, t = evaluar(3, control, dart, noches)
resultados.append(m)
tiempos.append(t)
display(estadisticas(m).round(4))
print(f"Ronda 3 · {m.subject_id.nunique()} personas · {len(m)} segmentos de prueba · ajuste de la rama {t['ajuste_rama_s']:.1f} s")

,F1 macro,Exactitud,Kappa
Modelo,,,
DART,0.8075,0.8522,0.8003
DART + CRNN,0.8181,0.8629,0.8150
V6 compacto,0.7453,0.7936,0.7200
DART + compacto sin HMM,0.8059,0.8433,0.7884
DART + compacto con HMM,0.8096,0.8464,0.7922


Ronda 3 · 7 personas · 15409 segmentos de prueba · ajuste de la rama 14.8 s


In [5]:
m, t = evaluar(4, control, dart, noches)
resultados.append(m)
tiempos.append(t)
display(estadisticas(m).round(4))
print(f"Ronda 4 · {m.subject_id.nunique()} personas · {len(m)} segmentos de prueba · ajuste de la rama {t['ajuste_rama_s']:.1f} s")

,F1 macro,Exactitud,Kappa
Modelo,,,
DART,0.8071,0.8629,0.8158
DART + CRNN,0.8128,0.8679,0.8221
V6 compacto,0.7522,0.8059,0.7419
DART + compacto sin HMM,0.8118,0.8629,0.8162
DART + compacto con HMM,0.8129,0.8642,0.8177


Ronda 4 · 7 personas · 14989 segmentos de prueba · ajuste de la rama 14.8 s


In [6]:
m, t = evaluar(5, control, dart, noches)
resultados.append(m)
tiempos.append(t)
display(estadisticas(m).round(4))
print(f"Ronda 5 · {m.subject_id.nunique()} personas · {len(m)} segmentos de prueba · ajuste de la rama {t['ajuste_rama_s']:.1f} s")

,F1 macro,Exactitud,Kappa
Modelo,,,
DART,0.7598,0.8274,0.7588
DART + CRNN,0.7599,0.8305,0.7623
V6 compacto,0.7520,0.8172,0.7468
DART + compacto sin HMM,0.7738,0.8380,0.7735
DART + compacto con HMM,0.7775,0.8427,0.7798


Ronda 5 · 6 personas · 13558 segmentos de prueba · ajuste de la rama 15.9 s


## Conjunto de las cinco rondas y tiempos

In [7]:
oof = pd.concat(resultados, ignore_index=True)
assert len(oof) == len(control) and not oof.duplicated(['record_id','epoch_index']).any()
display(pd.concat({'Ambas': estadisticas(oof), **{k: estadisticas(g) for k,g in oof.groupby('cohorte')}}).round(4))
display(pd.DataFrame(tiempos).set_index('ronda').round(2))
s_ref = json.loads((JUSTA/'tiempos-ronda-1/mediciones.json').read_text())['ajuste_dart_s']
t1 = tiempos[0]['ajuste_rama_s']
print(f'Primera ronda en serie, con entradas preparadas: DART medido previamente {s_ref:.1f} s + rama compacta {t1:.1f} s = {(s_ref+t1)/60:.2f} min.')
print('Matriz global de la combinación con HMM. Filas reales y columnas predichas.')
display(pd.DataFrame(confusion_matrix(oof.real,oof['DART + compacto con HMM'],labels=CLASES), index=CLASES,columns=CLASES))

F1 macro  Exactitud   Kappa
          Modelo                                              
Ambas     DART                       0.8010     0.8596  0.8074
          DART + CRNN                0.8086     0.8669  0.8171
          V6 compacto                0.7537     0.8202  0.7543
          DART + compacto sin HMM    0.8069     0.8624  0.8112
          DART + compacto con HMM    0.8096     0.8659  0.8157
Cassette  DART                       0.7768     0.9144  0.8233
          DART + CRNN                0.7949     0.9244  0.8450
          V6 compacto                0.6488     0.8582  0.7114
          DART + compacto sin HMM    0.7783     0.9129  0.8189
          DART + compacto con HMM    0.7789     0.9147  0.8225
Telemetry DART                       0.7881     0.8174  0.7474
          DART + CRNN                0.7930     0.8226  0.7534
          V6 compacto                0.7524     0.7909  0.7087
          DART + compacto sin HMM    0.7957     0.8235  0.7555
          DART + compacto con HMM    0.7990     0.8283  0.7612

,crnn_s,embeddings_cabezal_hmm_s,ajuste_rama_s,inferencia_rama_s,noches_prueba,parametros
ronda,,,,,,
1,9.32,5.72,15.03,1.06,11,13477
2,8.93,5.88,14.81,1.10,11,13477
3,9.12,5.69,14.81,1.09,12,13477
4,9.09,5.69,14.78,1.11,12,13477
5,9.95,5.95,15.90,0.99,10,13477


Primera ronda en serie, con entradas preparadas: DART medido previamente 420.7 s + rama compacta 15.0 s = 7.26 min.
Matriz global de la combinación con HMM. Filas reales y columnas predichas.


,N1,N2,N3,REM,W
N1,2878,733,18,440,790
N2,1397,21015,2172,724,720
N3,12,923,6411,2,44
REM,312,726,12,9168,159
W,720,180,10,46,26000


## Diferencias de F1 e inventario de particiones

Intervalos orientativos del 95 % al remuestrear personas dentro de cada cohorte. Se conservan juntas todas las noches de una persona y se comparan los modelos sobre la misma remuestra.

In [8]:
import numpy as np
rng = np.random.default_rng(42)
modelos = ['DART', 'DART + CRNN', 'DART + compacto con HMM']
remuestras = []
for _, cohorte in oof.groupby('cohorte'):
    matrices = np.array([[confusion_matrix(g.real, g[c], labels=CLASES) for c in modelos]
                         for _, g in cohorte.groupby('subject_id')])
    indices = rng.integers(len(matrices), size=(5000, len(matrices)))
    remuestras.append(matrices[indices].sum(axis=1))
cm = sum(remuestras)
den = cm.sum(axis=-2) + cm.sum(axis=-1)
f1 = np.divide(2*np.diagonal(cm, axis1=-2, axis2=-1), den, out=np.zeros(den.shape), where=den!=0).mean(axis=-1)
scores = estadisticas(oof)['F1 macro']
for i, modelo in enumerate(modelos[:-1]):
    diferencia = scores[modelos[-1]] - scores[modelo]
    inferior, superior = np.quantile(f1[:, -1] - f1[:, i], [.025, .975])
    print(f'Compacto con DART frente a {modelo}: {100*diferencia:+.2f} puntos de F1 (IC 95 % {100*inferior:+.2f} a {100*superior:+.2f}).')
particiones = []
for ronda in sorted(oof.ronda.unique()):
    for uso, seleccion in [('Entrenamiento',oof.ronda.ne(ronda)), ('Prueba',oof.ronda.eq(ronda))]:
        for cohorte, g in oof[seleccion].groupby('cohorte'):
            particiones.append([ronda,uso,cohorte,g.subject_id.nunique(),g.record_id.nunique(),len(g)])
display(pd.DataFrame(particiones, columns=['Ronda','Uso','Cohorte','Personas','Noches','Segmentos']).set_index(['Ronda','Uso','Cohorte']))

Compacto con DART frente a DART: +0.86 puntos de F1 (IC 95 % +0.18 a +1.52).
Compacto con DART frente a DART + CRNN: +0.10 puntos de F1 (IC 95 % -0.78 a +0.90).


Personas  Noches  Segmentos
Ronda Uso           Cohorte                               
1     Entrenamiento Cassette          9       9      24882
                    Telemetry        18      36      34837
      Prueba        Cassette          3       3       8039
                    Telemetry         4       8       7854
2     Entrenamiento Cassette          9       9      24681
                    Telemetry        18      36      35168
      Prueba        Cassette          3       3       8240
                    Telemetry         4       8       7523
3     Entrenamiento Cassette         10      10      27327
                    Telemetry        17      34      32876
      Prueba        Cassette          2       2       5594
                    Telemetry         5      10       9815
4     Entrenamiento Cassette         10      10      27431
                    Telemetry        17      34      33192
      Prueba        Cassette          2       2       5490
                    Telemetry         5      10       9499
5     Entrenamiento Cassette         10      10      27363
                    Telemetry        18      36      34691
      Prueba        Cassette          2       2       5558
                    Telemetry         4       8       8000

## Un solo LightGBM: DART + CRNN + HMM

Se retira el segundo clasificador de árboles de la rama V6. Cada CRNN ya tiene una salida softmax con cinco probabilidades. Se aplica el HMM a esas probabilidades noche por noche y después se mezclan con DART (75 % DART y 25 % CRNN con HMM). El HMM usa solo las etiquetas de entrenamiento para estimar sus transiciones. Comparamos la CRNN compacta en cinco rondas y la CRNN original del compañero en la única ronda con pesos completos. El cálculo reutiliza los pesos guardados.

In [9]:
import zipfile
import tensorflow as tf
from time import perf_counter
from scripts.evaluar_v6_compacto import bloques, posterior, transiciones, PESO_DART
from sleep_staging.datasets import VALID_STAGES
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score

carpeta_compacta = root/'.sleep-cv/v6-compacto/37ad050a38624756'
referencia = root/'data/referencias/hibrido_v6/modelo_crnn_maestro_optimo_v7.keras'
with zipfile.ZipFile(referencia) as archivo:
    arquitectura = json.loads(archivo.read('config.json'))
arquitectura.pop('compile_config', None)
original = tf.keras.models.model_from_json(json.dumps(arquitectura))
original.load_weights(JUSTA/'v6-1/encoder.weights.h5')
softmax_original = original.get_layer('softmax_clasificador')
assert original.layers[-1] is softmax_original and softmax_original.input.shape[-1] == 64
orden = [list(VALID_STAGES).index(c) for c in CLASES]

def puntuar_rama(probabilidades, g, matriz, prior):
    assert np.all(np.isfinite(probabilidades)) and np.allclose(probabilidades.sum(1), 1, atol=1e-5)
    suavizadas = posterior(probabilidades.astype('float64'), matriz, prior)[g.epoch_index]
    assert np.isfinite(suavizadas).all() and np.allclose(suavizadas.sum(1), 1)
    return CLASES[(PESO_DART*dart[g.index] + (1-PESO_DART)*suavizadas).argmax(1)]

directo = control.copy()
tiempo_hmm = []
comprobado = False
for ronda in sorted(directo.ronda.unique()):
    test = directo.ronda.eq(ronda)
    assert set(directo.loc[test,'subject_id']).isdisjoint(directo.loc[~test,'subject_id'])
    inicio = perf_counter()
    matriz, prior = transiciones(directo.loc[~test])
    tiempo_hmm.append(perf_counter() - inicio)
    ruta = carpeta_compacta/str(ronda)
    red = tf.keras.models.load_model(ruta/'crnn.keras', compile=False)
    with np.load(ruta/'hmm.npz') as previo:
        np.testing.assert_allclose(matriz, previo['A'])
        np.testing.assert_allclose(prior, previo['prior'])
    for rid, g in directo[test].groupby('record_id'):
        x = noches[rid]
        prob = red.predict(bloques(x), batch_size=128, verbose=0).reshape(-1, len(CLASES))[:len(x)]
        directo.loc[g.index,'Compacta directa'] = puntuar_rama(prob, g, matriz, prior)
        if ronda == 1:
            # Las 192 columnas guardadas son [anterior, actual, siguiente].
            emb = np.load(JUSTA/'v6-1'/f'{rid}-192.npy', mmap_mode='r')[:,64:128]
            prob_original = softmax_original(emb, training=False).numpy()
            if not comprobado:
                signal = np.load(JUSTA/f'{rid}-signal.npy', mmap_mode='r')
                centro = len(signal)//2
                ventana = signal[None, centro-7:centro+8]
                entrada = {'seq_eeg': ventana[..., :2], 'seq_eog': ventana[..., 2:3], 'seq_emg': ventana[..., 3:]}
                np.testing.assert_allclose(original(entrada, training=False).numpy()[0], prob_original[centro], atol=.003)
                comprobado = True
            directo.loc[g.index,'Original directa'] = puntuar_rama(prob_original[:,orden], g, matriz, prior)
assert comprobado and directo['Compacta directa'].notna().all()
assert directo['Original directa'].notna().equals(directo.ronda.eq(1))
print('Segmentos con la CRNN compacta:', len(directo), '· con la CRNN original:', directo['Original directa'].count())

Segmentos con la CRNN compacta: 75612 · con la CRNN original: 15893


In [10]:
claves = ['record_id','epoch_index','subject_id','real','ronda']
pares = directo.merge(oof[claves+['DART + compacto con HMM']], on=claves, validate='one_to_one')
assert len(pares)==len(control)
def comparar_directa(grupo, columnas):
    filas = []
    for col in columnas:
        y, pred = grupo.real, grupo[col]
        filas.append([col,len(grupo),f1_score(y,pred,labels=CLASES,average='macro'),accuracy_score(y,pred),cohen_kappa_score(y,pred)])
    return pd.DataFrame(filas,columns=['Modelo','Segmentos','F1 macro','Exactitud','Kappa']).set_index('Modelo')
columnas = ['DART','DART + compacto con HMM','Compacta directa']
display(pd.concat({'Cinco rondas': comparar_directa(pares,columnas),
                   'Primera ronda': comparar_directa(pares[pares.ronda.eq(1)], columnas+['Original directa'])}).round(4))
display(pd.concat({c: comparar_directa(g,columnas) for c,g in pares.groupby('cohorte')}).round(4))
print(f'Primera ronda, ajuste en serie desde entradas preparadas: DART {s_ref:.2f} s + CRNN compacta {tiempos[0]["crnn_s"]:.2f} s + HMM {tiempo_hmm[0]:.3f} s.')

Segmentos  F1 macro  Exactitud   Kappa
              Modelo                                                         
Cinco rondas  DART                         75612    0.8010     0.8596  0.8074
              DART + compacto con HMM      75612    0.8096     0.8659  0.8157
              Compacta directa             75612    0.8075     0.8670  0.8163
Primera ronda DART                         15893    0.8194     0.8745  0.8262
              DART + compacto con HMM      15893    0.8284     0.8844  0.8395
              Compacta directa             15893    0.8263     0.8837  0.8382
              Original directa             15893    0.8280     0.8833  0.8382

Segmentos  F1 macro  Exactitud   Kappa
          Modelo                                                         
Cassette  DART                         32921    0.7768     0.9144  0.8233
          DART + compacto con HMM      32921    0.7789     0.9147  0.8225
          Compacta directa             32921    0.7737     0.9146  0.8216
Telemetry DART                         42691    0.7881     0.8174  0.7474
          DART + compacto con HMM      42691    0.7990     0.8283  0.7612
          Compacta directa             42691    0.7978     0.8302  0.7624

Primera ronda, ajuste en serie desde entradas preparadas: DART 420.66 s + CRNN compacta 9.32 s + HMM 0.030 s.
